# Exercise — Improve a Topic-Extraction Prompt

**Task:** a prompt that extracts the topics from a passage of text. The starter is deliberately naive:

```python
def run_prompt(prompt_inputs):
    prompt = f"""
What topics are in here?

{prompt_inputs["content"]}
"""
    ...
```

**Goal:** take the eval score from ~2 to ~7 by applying the section's techniques.

The instructor iterated in stages — first just **clear & direct + a specific output format** (*"…into a JSON array of strings"*), which already does most of the work — then landed on this **final** version that also adds an XML wrapper and process steps:

```
Extract key topics mentioned from a passage of text from a scholarly journal into a JSON array of strings.

<text>
{content}
</text>

Follow these steps:
1. Closely examine the provided text
2. Identify each topic mentioned
3. Add each topic to a JSON array
4. Respond with the JSON array. Do not provide any other text or commentary
```

Notably it uses **process steps** rather than a one-shot example — the two are interchangeable ways to be specific. This notebook runs the naive baseline, then this final solution.

> **Notes:** This exercise uses a different task and input (`content`) than the meal-plan lessons, so it gets its **own** dataset — written to `dataset-exercise.json` (the shared `dataset.json` is left untouched). The course's exact `task_description` was off-screen; a sensible topic-extraction one is used here. Runs on the same Haiku harness.

## Load the harness

In [ ]:
import sys
import os
from dotenv import find_dotenv

REPO_ROOT = os.path.dirname(find_dotenv())
SECTION = os.path.join(REPO_ROOT, "building-with-the-claude-api", "03-prompt-engineering")
if SECTION not in sys.path:
    sys.path.insert(0, SECTION)

from prompt_evaluator import PromptEvaluator, add_user_message, chat

DATASET_EX = os.path.join(SECTION, "dataset-exercise.json")
evaluator = PromptEvaluator(max_concurrent_tasks=3)
print("Harness loaded. Exercise dataset ->", DATASET_EX)

Harness loaded. Exercise dataset -> c:\Development\anthropic-cert\building-with-the-claude-api\03-prompt-engineering\dataset-exercise.json


## Generate the exercise dataset

A topic-extraction task with a single `content` input. Four cases so there's enough signal to see the score move.

In [ ]:
dataset = evaluator.generate_dataset(
    # Aligned with the improved prompt's goal: comprehensive extraction into a JSON array.
    # (A vaguer "identify the MAIN topics" framing makes the generator write criteria that
    #  PENALIZE listing subtopics/details — which then punishes the JSON-array prompt. See the note below.)
    task_description="Extract the key topics mentioned in a passage of text into a JSON array of short topic strings",
    prompt_inputs_spec={
        "content": "One paragraph of text from a scholarly journal written in English",
    },
    output_file=DATASET_EX,
    num_cases=4,
)

dataset

## Baseline — the naive starter prompt

Establish the low score first.

In [ ]:
def run_prompt_baseline(prompt_inputs):
    prompt = f"""
What topics are in here?

{prompt_inputs["content"]}
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


baseline = evaluator.run_evaluation(
    run_prompt_function=run_prompt_baseline,
    dataset_file=DATASET_EX,
    json_output_file=os.path.join(SECTION, "output-06-exercise-baseline.json"),
    html_output_file=os.path.join(SECTION, "output-06-exercise-baseline.html"),
)

Graded 1/4 test cases
Graded 2/4 test cases
Graded 3/4 test cases
Graded 4/4 test cases
Average score: 7.25


## Improved — the instructor's final solution

The instructor iterated to this, stacking four techniques (but *not* a one-shot example):

1. **Clear & direct** — "Extract key topics..."
2. **Specific format** — "...into a JSON array of strings"
3. **XML tags** — the passage wrapped in `<text>`
4. **Process steps** — a numbered "Follow these steps" that walks Claude through examine → identify → collect → respond, ending with "no commentary"

Process steps stand in for the example here: rather than *showing* an ideal output, they *tell* Claude the procedure and forbid extra prose.

In [4]:
def run_prompt(prompt_inputs):
    prompt = f"""
Extract key topics mentioned from a passage of text from a scholarly journal into a JSON array of strings.

<text>
{prompt_inputs["content"]}
</text>

Follow these steps:
1. Closely examine the provided text
2. Identify each topic mentioned
3. Add each topic to a JSON array
4. Respond with the JSON array. Do not provide any other text or commentary
"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


improved = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file=DATASET_EX,
    json_output_file=os.path.join(SECTION, "output-06-exercise-improved.json"),
    html_output_file=os.path.join(SECTION, "output-06-exercise-improved.html"),
)

Graded 1/4 test cases
Graded 2/4 test cases
Graded 3/4 test cases
Graded 4/4 test cases
Average score: 5.5


## Compare — and a real eval-design lesson

Expect the improved (JSON-array + process-steps) prompt to now beat the naive baseline.

**Why the `task_description` above had to change.** On the first pass it was the vaguer *"Identify and list the **main** topics."* From that, the dataset generator wrote `solution_criteria` that actively **penalize over-listing** — e.g. *"subtopics are components of one main subject, not separate main topics"*, *"distinguish main topics from supporting details."* But the instructor's improved prompt says *"identify **each** topic → add **each** to the array,"* which over-extracts every subtopic — precisely what those criteria dock. Result: the "improved" prompt scored **lower** (5.5) than the vague baseline (7.25).

The grader only ever rewards **what the criteria say**. A prompt is only "better" *relative to the criteria it's graded against*, and the criteria come from the `task_description`. When the task framing and the prompt's behavior disagree, a genuinely more useful prompt can score worse. Aligning the `task_description` with the intended output (comprehensive extraction into a JSON array) makes the criteria reward what the improved prompt does — and the expected ~2→~7 story reappears.

This is the whole reason prompt evaluation and prompt engineering are taught together: iterate the prompt **and** make sure the eval is measuring the thing you actually want.